In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "py_src")
sys.path.insert(0, "../py_src")
sys.path.insert(0, "../../py_src")

from voxeloo.notebooks.imports import *

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dataclasses import dataclass
from perlin_numpy import generate_fractal_noise_2d, generate_fractal_noise_3d
from poisson_disc import Bridson_sampling
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d
from scipy.ndimage import gaussian_filter as blur
from tqdm.notebook import tqdm
from typing import List, Set, Dict
from _voxeloo_cpp_ext import BlockList, voronoi, voxels_to_mesh, biomes

In [ ]:
@dataclass
class BiomeGraph:
    sites: np.array
    neighbors: Dict[int, Set[int]]
        
    def __len__(self):
        return self.sites.shape[0]

In [ ]:
def outside(shape, point):
    return np.any(np.maximum(-point, point - shape) > 0)

In [ ]:
def uniform_points(shape, radius):
    return Bridson_sampling(np.array(shape), radius=radius, k=30)

In [ ]:
def external_ridge_2d(shape, voronoi, ridge):
    p, q = voronoi.ridge_vertices[ridge]
    return outside(shape, voronoi.vertices[p]) and outside(shape, voronoi.vertices[q])

def generate_2d_biomes(shape=(256, 256), radius=30):
    points = uniform_points(shape, radius)
    
    # Build a neighborhood graph out of the sites.
    voronoi = Voronoi(points)
    n = voronoi.points.shape[0]
    graph = BiomeGraph(points, {})
    for i in range(n):
        graph.neighbors[i] = set()
    for i in range(voronoi.ridge_points.shape[0]):
        if external_ridge_2d(shape, voronoi, i):
            continue
        u, v = voronoi.ridge_points[i]
        graph.neighbors[u].add(v)
        graph.neighbors[v].add(u)

    return graph

In [ ]:
def rasterize_biomes_2d(shape, sites, colors):
    indices = voronoi.rasterize_2d(sites, shape)
    return colors[indices]

In [ ]:
def external_ridge_3d(shape, voronoi, ridge):
    for i in voronoi.ridge_vertices[ridge]:
        if not outside(shape, voronoi.vertices[i]):
            return False
    return True

def generate_3d_biomes(shape=(256, 256, 256), radius=30):
    points = uniform_points(shape, radius)
    
    # Build a neighborhood graph out of the sites.
    graph = BiomeGraph(points, {})
    for i in range(points.shape[0]):
        graph.neighbors[i] = set()
    voronoi = Voronoi(points)
    for i in range(voronoi.ridge_points.shape[0]):
        if external_ridge_3d(shape, voronoi, i):
            continue
        u, v = voronoi.ridge_points[i]
        graph.neighbors[u].add(v)
        graph.neighbors[v].add(u)
        
    return graph

In [ ]:
def rasterize_biomes_3d(shape, sites, colors):
    indices = voronoi.rasterize_3d(sites, shape)
    return colors[indices]

In [ ]:
def color_index(n):
    return np.random.randint(0, 255, size=(n, 3))

In [ ]:
def compact_color_index(n):
    colors = color_index(n)
    return sum([
        0xff,
        0xff * (colors[:, 0].astype(np.uint32) << 8),
        0xff * (colors[:, 1].astype(np.uint32) << 16),
        0xff * (colors[:, 2].astype(np.uint32) << 32),
    ])

### Generate random biome sites

In [ ]:
x_dim, y_dim, z_dim = 512, 256, 512

In [ ]:
%%time 
biomes = generate_2d_biomes(shape=(x_dim, z_dim), radius=20)

In [ ]:
%%time

colors = np.random.randint(0, 255, size=(len(biomes), 3))
render = rasterize_biomes_2d(
    shape=(x_dim, z_dim),
    sites=biomes.sites,
    colors=colors,
)

In [ ]:
plt.imshow(render)
plt.show()

### Assign sites to biome types

In [ ]:
def random_walk(biomes, steps, valid, fn):
    node = random.randint(0, len(biomes) - 1)    
    for i in range(steps):
        fn(node)
        goto = [node for node in biomes.neighbors[node] if valid(node)]
        if goto:
            node = random.choice(list(goto))
        else:
            break

In [ ]:
def random_walk_with_momentum(biomes, steps, noise, valid, fn):
    past = np.array([0, 0]) 
    node = random.randint(0, len(biomes) - 1)    
    for i in range(steps):
        fn(node)
        goto = [node for node in biomes.neighbors[node] if valid(node)]
        if goto:
            dirs = [(i, biomes.sites[i] - biomes.sites[node]) for i in goto]
            dirs.sort(key=lambda d: noise * np.random.random() - past.dot(d[1]))
            node, past = dirs[0]
        else:
            break

#### Add grass

In [ ]:
assignments = {
    i: "grass" for i in range(len(biomes))
}

#### Add mountains

In [ ]:
def visit_mountain(node):
    assignments[node] = "mountain"

for _ in range(12):
    random_walk(
        biomes,
        10,
        lambda node: assignments[node] == "grass",
        visit_mountain,
    )

#### Add canyons

In [ ]:
def visit_canyon(node):
    assignments[node] = "canyon"

for _ in range(6):
    random_walk_with_momentum(
        biomes,
        10,
        0.1,
        lambda node: assignments[node] == "grass",
        visit_canyon,
    )

#### Add groves

In [ ]:
def visit_grove(node):
    assignments[node] = "grove"

for _ in range(15):
    random_walk(
        biomes,
        random.randint(1, 10),
        lambda node: assignments[node] == "grass",
        visit_grove,
    )

#### Visualize biomes with a color render

In [ ]:
biome_colors = {
    "grass": [80, 180, 80],
    "mountain": [140, 70, 60],
    "canyon": [80, 80, 120],
    "grove": [80, 120, 80],
}

In [ ]:
%%time

render = rasterize_biomes_2d(
    shape=(x_dim, z_dim),
    sites=biomes.sites,
    colors=np.array([
        biome_colors[kind] for _, kind in sorted(assignments.items())
    ]),    
)

In [ ]:
plt.imshow(render)
plt.show()

### Generate height map from biomes

In [ ]:
def noise_to_heights(noise, avg, std):
    return (avg + std * (noise - noise.mean()) / noise.std()).astype(np.uint32)

In [ ]:
def gen_base_heights(shape):
    return noise_to_heights(
        noise=generate_fractal_noise_2d(shape, (1, 1), 3),
        avg=128,
        std=8,
    )

In [ ]:
def gen_grass_heights(shape):
    return gen_base_heights(shape) + noise_to_heights(
        noise=generate_fractal_noise_2d(shape, (4, 4), 4),
        avg=0,
        std=2,
    )

In [ ]:
def gen_mountain_heights(shape):
    return gen_base_heights(shape) + noise_to_heights(
        noise=generate_fractal_noise_2d(shape, (8, 8), 5),
        avg=24,
        std=16,
    )

In [ ]:
def gen_canyon_heights(shape):
    return gen_base_heights(shape) + noise_to_heights(
        noise=generate_fractal_noise_2d(shape, (8, 8), 3),
        avg=-16,
        std=4,
    )

In [ ]:
def gen_grove_heights(shape):
    return gen_base_heights(shape) + noise_to_heights(
        noise=generate_fractal_noise_2d(shape, (16, 16), 2),
        avg=0,
        std=1,
    )

In [ ]:
def biome_mask(biome):
    return rasterize_biomes_2d(
        shape=(x_dim, z_dim),
        sites=biomes.sites,
        colors=np.array([
            1.0 if kind == biome else 0.0 for _, kind in sorted(assignments.items())
        ]),    
    )

In [ ]:
def blend_heights(mask, src, dst):
    return ((1 - mask) * src + mask * dst).astype(np.uint32)

#### Initialize to the grass biome

In [ ]:
heights = gen_grass_heights(shape=(x_dim, z_dim))

#### Blend in the mountain heights

In [ ]:
mountain_mask = blur(biome_mask("mountain"), 4.0, order=0)
mountain_heights = gen_mountain_heights(shape=(x_dim, z_dim))

heights = blend_heights(
    mask=mountain_mask,
    src=heights,
    dst=mountain_heights,
)

#### Blend in tree heights

In [ ]:
grove_mask = blur(biome_mask("grove"), 1.0, order=0)
grove_heights = gen_grove_heights(shape=(x_dim, z_dim))

heights = blend_heights(
    mask=grove_mask,
    src=heights,
    dst=grove_heights,
)

#### Blend in the canyon heights

In [ ]:
canyon_mask = blur(biome_mask("canyon"), 1.0, order=0)
canyon_heights = gen_canyon_heights(shape=(x_dim, z_dim))

heights = blend_heights(
    mask=canyon_mask,
    src=heights,
    dst=canyon_heights,
)

### Create 3D voxels from the height map

In [ ]:
voxels = np.zeros(
    shape=[x_dim, y_dim, z_dim],
    dtype=np.uint32,
)

In [ ]:
%%time 

for x in range(x_dim):
    for z in range(z_dim):
        h = heights[x, z]
        voxels[x, 0:h, z] = 0xffffffff

In [ ]:
visualize_3d_window(
    voxels_to_mesh(voxels)
)

### Set voxel types

In [ ]:
 voxel_kinds = {
     "stone": 1,
     "grass": 2,
     "dirt": 3,
     "cobblestone": 4,
     "bedrock": 7,
     "gravel": 13,
     "oak": 17,
     "mossy_cobblestone": 48,
     "obsidian": 49,
     "leaf": 87,
     "mushroom": 88,
     "spruce": 89, 
     "light_leaf": 90,
     "granite": 103,
 }

In [ ]:
 voxel_colors = {
    "stone": 0xbababaff,
    "grass": 0x50b450ff,
    "dirt": 0x8f635dff,
    "cobblestone": 0x70706fff,
    "bedrock": 0x424647ff,
    "gravel": 0x85837eff,
    "oak": 0x916b39ff,
    "mossy_cobblestone": 0x6e8069ff,
    "obsidian": 0x171c1fff,
    "leaf": 0x5e8a29ff,
    "mushroom": 0xb54038ff,
    "spruce": 0x665135ff,
    "light_leaf": 0x6bb347ff,
    "granite": 0xbd856aff,
 }

In [ ]:
%%time

biome_map = rasterize_biomes_2d(
    shape=(x_dim, z_dim),
    sites=biomes.sites,
    colors=np.array([
        kind for i, kind in sorted(assignments.items())
    ]),    
)

In [ ]:
def clamp(x, lo, hi):
    return min(max(x, lo), hi)

def slow_random(x, z):
    return random.Random((x // 8) * (z // 8))

def underground_column(x, y, z):
    obsidian = clamp(slow_random(x, z).randint(8, 48), 0, y - 8)
    bedrock = clamp(obsidian + slow_random(x, z).randint(32, 72), obsidian, y - 8)
    voxels[x, 0:obsidian, z] = voxel_kinds["obsidian"]
    voxels[x, obsidian:bedrock, z] = voxel_kinds["bedrock"]
    voxels[x, bedrock:y, z] = voxel_kinds["stone"]
    
for x in range(x_dim):
    for z in range(z_dim):
        biome = biome_map[x, z]
        h = heights[x, z]
        
        if biome == "grass":
            d = max(0, h - random.randint(2, 20))
            voxels[x, h - 1, z] = voxel_kinds["grass"]
            voxels[x, d:h - 1, z] = voxel_kinds["dirt"]
            underground_column(x, d, z)
        elif biome == "mountain":
            d = max(0, h - random.randint(2, 5))
            s = 128 + random.randint(0, 8)
            if h < s:
                voxels[x, h - 1, z] = voxel_kinds["grass"]
                voxels[x, d:h - 1, z] = voxel_kinds["dirt"]
            else:
                voxels[x, d:h, z] = voxel_kinds["dirt"]
            underground_column(x, d, z)
        elif biome == "canyon":
            d = max(0, h - random.randint(1, 4))
            voxels[x, d:h, z] = random.choice([
                voxel_kinds["gravel"],
                voxel_kinds["granite"],
            ])
            underground_column(x, d, z)
        elif biome == "grove":
            d = max(0, h - random.randint(8, 24))
            voxels[x, h - 1, z] = voxel_kinds["grass"]
            voxels[x, d:h - 1, z] = voxel_kinds["dirt"]
            underground_column(x, d, z)

In [ ]:
def to_colored_voxels(voxels):
    ret = np.zeros(voxels.shape, dtype=np.uint32)
    for key, val in voxel_kinds.items():
        ret[voxels == val] = voxel_colors[key]
    return ret

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(voxels))
)

### Add caves

In [ ]:
%%time 
biomes_3d = generate_3d_biomes(shape=(x_dim, y_dim, z_dim), radius=10)

In [ ]:
%%time

render = rasterize_biomes_3d(
    shape=(x_dim, y_dim, z_dim),
    sites=biomes_3d.sites,
    colors=compact_color_index(len(biomes_3d.sites)),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(render)
)

In [ ]:
def cave_walk(biomes, steps, turn_prob, fn):
    past = np.array([0, 0, 0])
    node = random.randint(0, len(biomes) - 1)    
    for i in range(steps):
        fn(node)
        goto = [node for node in biomes.neighbors[node]]
        if goto:
            dirs = [(i, biomes.sites[i] - biomes.sites[node]) for i in goto]
            dirs.sort(key=lambda d: -past.dot(d[1]))
            if random.random() < turn_prob:
                node, past = random.choice(dirs)
            else:
                node, past = dirs[0]
        else:
            break

In [ ]:
caves = set()

def visit_cave(node):
    caves.add(node)

for _ in range(20):
    cave_walk(
        biomes_3d,
        50,
        0.2,
        visit_cave,
    )

In [ ]:
%%time

colors = np.array([0 if i in caves else 1 for i in range(len(biomes_3d.sites))])
cave_voxels = rasterize_biomes_3d(
    shape=(x_dim, y_dim, z_dim),
    sites=biomes_3d.sites,
    colors=colors,
)

In [ ]:
# Chop caves off above a certain height.
cave_voxels[:, 144:, :] = 1

In [ ]:
def smooth(tensor, std=4.0, t=0.8):
    ret = blur(tensor.astype(np.float64), std)
    ret[ret < t] = 0
    ret[ret != 0] = 1
    return ret

In [ ]:
%%time

cave_voxels = smooth(smooth(cave_voxels, std=6, t=0.85), std=4, t=0.25)

In [ ]:
render = np.array(cave_voxels, dtype=np.uint32)
render[cave_voxels == 1] = 0
render[cave_voxels == 0] = 0xffffffff

visualize_3d_window(
    voxels_to_mesh(render)
)

In [ ]:
voxels[cave_voxels == 0] = 0

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(voxels))
)

### Add surface features

In [ ]:
def place_features(locations, samples):
    feature_voxels = np.zeros(voxels.shape, dtype=np.uint32)
    
    for x, z in locations:
        y = heights[x, z]

        feature = random.choice(samples)
        w = feature.shape[0]
        h = feature.shape[1]
        d = feature.shape[2]

        # Work out the location to place the tree.
        x0 = x - w // 2
        x1 = x0 + w
        y0 = y
        y1 = y + h 
        z0 = z + d // 2
        z1 = z0 + d

        # Make sure that the feature placement is inside the bounds.
        if x0 < 0 or x1 >= feature_voxels.shape[0]:
            continue
        if y0 < 0 or y1 >= feature_voxels.shape[1]:
            continue
        if z0 < 0 or z1 >= feature_voxels.shape[2]:
            continue
            
        # Make sure that the feature placement has no intersections.
        mask = (voxels[x0:x1, y0:y1, z0:z1] == 0).astype(int)
        mask = mask - (feature[:, :, :] != 0).astype(int)
        if np.any(mask < 0):
            continue
            
        # Make sure that the feature placement is supported.   
        support = (voxels[x0:x1, y0 - 1, z0:z1] != 0).astype(int)
        support = support - (feature[:, 0, :] != 0).astype(int)
        if np.any(support < 0):
            continue
            
        feature_voxels[x0:x1, y0:y1, z0:z1] = feature 
        
    return feature_voxels

#### Add trees

In [ ]:
def gen_tree(h, wood="oak", leaf="leaf"):
    wood = voxel_kinds[wood]
    leaf = voxel_kinds[leaf]
    ret = np.zeros((5, h + 5, 5))
    ret[1:4, h:h + 5, 1:4] = leaf
    ret[1:4, h + 1:h + 4, 0:5] = leaf
    ret[0:5, h + 1:h + 4, 1:4] = leaf
    ret[0:5, h + 2:h + 3, 0:5] = leaf
    ret[2, 0:h, 2] = wood
    return ret

In [ ]:
%%time

tree_points = []

# Restrict to points within the grove biome.
cp = uniform_points((512, 512), 4).astype(np.uint32)
cp = cp[biome_mask("grove")[cp[:, 0], cp[:, 1]] == 1, :]
tree_points.extend(cp.tolist())

# Add in some points from the grass biome.
cp = uniform_points((512, 512), 24).astype(np.uint32)
cp = cp[biome_mask("grass")[cp[:, 0], cp[:, 1]] == 1, :]
tree_points.extend(cp.tolist())

In [ ]:
%%time 
tree_voxels = place_features(
    locations=tree_points,
    samples=[
        gen_tree(4),
        gen_tree(5),
        gen_tree(6),
        gen_tree(7),
        gen_tree(8),
        gen_tree(4, "spruce", "light_leaf"),
        gen_tree(5, "spruce", "light_leaf"),
        gen_tree(6, "spruce", "light_leaf"),
    ],
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(tree_voxels))
)

In [ ]:
voxels[tree_voxels != 0] = tree_voxels[tree_voxels != 0]

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(voxels))
)

#### Add mushrooms

In [ ]:
def gen_mushroom_patch(shape, radius=2):
    cp = uniform_points(shape, radius).astype(np.uint32)
    ret = np.zeros(shape)[:, np.newaxis, :].astype(np.uint32)
    ret[cp[:, 0], 0, cp[:, 1]] = voxel_kinds["mushroom"]
    return ret

In [ ]:
%%time

mushroom_points = []

# Restrict to points within the grove biome.
cp = uniform_points((512, 512), 8).astype(np.uint32)
cp = cp[biome_mask("grove")[cp[:, 0], cp[:, 1]] == 1, :]
mushroom_points.extend(cp.tolist())

In [ ]:
%%time 
mushroom_voxels = place_features(
    locations=mushroom_points,
    samples=[
        gen_mushroom_patch((3, 3)),
        gen_mushroom_patch((4, 4)),
        gen_mushroom_patch((5, 5)),
        gen_mushroom_patch((7, 7)), 
    ],
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(mushroom_voxels))
)

In [ ]:
voxels[mushroom_voxels != 0] = mushroom_voxels[mushroom_voxels != 0]

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(voxels))
)

#### Add henges

In [ ]:
def gen_henges(shape, radius=3):
    w, d = shape
    cp = uniform_points(shape, radius).astype(np.uint32)
    ret = np.zeros((w, 5, d)).astype(np.uint32)
    for x, y in cp:
        a = random.randint(1, 2)
        b = random.randint(0, 3)
        ret[x, 0:a, y] = voxel_kinds["mossy_cobblestone"]
        ret[x, a:b, y] = voxel_kinds["cobblestone"]
    return ret

In [ ]:
%%time

henge_points = []

# Restrict to points within the grove biome.
cp = uniform_points((512, 512), 8).astype(np.uint32)
cp = cp[biome_mask("grass")[cp[:, 0], cp[:, 1]] == 1, :]
henge_points.extend(cp.tolist())

In [ ]:
%%time 
henge_voxels = place_features(
    locations=henge_points,
    samples=[
        gen_henges((8, 8), radius=3),
        gen_henges((16, 16), radius=4),
    ],
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(henge_voxels))
)

In [ ]:
voxels[henge_voxels != 0] = henge_voxels[henge_voxels != 0]

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_colored_voxels(voxels))
)

### Save to volume block files

In [ ]:
%%time

x_blocks = x_dim // 32
y_blocks = y_dim // 32
z_blocks = z_dim // 32

for bx in tqdm(range(x_blocks)):
    for by in range(y_blocks):
        for bz in range(z_blocks):
            block = biomes.VolumeBlock_U32()
            for x in range(32):
                vx = 32 * bx + x
                for y in range(32):
                    vy = 32 * by + y
                    for z in range(32):
                        vz = 32 * bz + z
                        block[x, y, z] = voxels[vx, vy, vz]
            blob = block.dumps()
            path = f"../../../voxeloo-web/data/world/block_{bx}_{by}_{bz}"
            with open(path, "w") as f:
                f.write(blob)